# Phase 5: Advanced Techniques + Ablation + LLM Comparison
**Visual Product Search Engine — 2026-04-24**  
**Researcher:** Mark Rodrigues

## Research Question
Phase 4 found 85.3% of failures are close misses (correct in top-5, gap < 0.05).
Anthony found text_prompt R@10=1.000 — the correct product is ALWAYS in text top-10.

**Hypothesis:** Use visual to scope candidates, use text to precision-rank them.
Two-stage and three-stage pipelines should push R@1 from 0.683 above 0.80.

In [ ]:
import sys, os, re
sys.path.insert(0, '..')
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import json, time, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

PROJECT = Path('..')
PROC    = PROJECT / 'data' / 'processed'
RES     = PROJECT / 'results'
CACHE   = PROC / 'emb_cache'

EVAL_N = 300; K_TOP = 20
print('Setup complete')

In [ ]:
# Load data and embeddings
gallery_df = pd.read_csv(PROC / 'gallery.csv')
query_df   = pd.read_csv(PROC / 'query.csv')
eval_pids  = gallery_df['product_id'].values[:EVAL_N]
g_df = gallery_df[gallery_df['product_id'].isin(eval_pids)].reset_index(drop=True)
q_df = query_df[query_df['product_id'].isin(eval_pids)].reset_index(drop=True)

gallery_cats = g_df['category2'].values
query_cats   = q_df['category2'].values
q_pids = q_df['product_id'].values
g_pids = g_df['product_id'].values

def norm(x): n = np.linalg.norm(x, axis=1, keepdims=True); return x / np.maximum(n, 1e-8)
def load_emb(name): return norm(np.load(CACHE / f'{name}.npy').astype(np.float32))
def recall_at_k(indices, qp, gp, k): return float(sum(qp[i] in gp[indices[i][:k]] for i in range(len(indices)))) / len(indices)
def evaluate(indices, qp, gp, label=''):
    res = {f'R@{k}': recall_at_k(indices, qp, gp, k) for k in [1,5,10,20]}
    if label: print(f'  {label}: R@1={res["R@1"]:.4f}  R@5={res["R@5"]:.4f}  R@10={res["R@10"]:.4f}')
    return res

g_clip  = load_emb('clip_b32_gallery')
q_clip  = load_emb('clip_b32_query')
g_color = load_emb('color48_gallery')
q_color = load_emb('color48_query')
g_text  = load_emb('clip_b32_text_gallery')
q_text  = load_emb('clip_b32_text_query')
print(f'CLIP visual:  gallery={g_clip.shape}, query={q_clip.shape}')
print(f'Color 48D:    gallery={g_color.shape}, query={q_color.shape}')
print(f'CLIP text:    gallery={g_text.shape}, query={q_text.shape}')

## 5.M.1 Baseline + Text Sanity Check

In [ ]:
# Phase 3 champion: cat + CLIP + color alpha=0.4
GLOBAL_ALPHA = 0.40

def cat_rerank_search(q_vis, g_vis, q_col, g_col, qcats, gcats, alpha=GLOBAL_ALPHA, topk=K_TOP):
    all_top = []
    for i in range(len(q_vis)):
        cat = qcats[i]
        cidx = np.where(gcats == cat)[0]
        clip_s = g_vis[cidx] @ q_vis[i]
        color_s = g_col[cidx] @ q_col[i]
        fused = (1 - alpha) * clip_s + alpha * color_s
        order = np.argsort(-fused)[:topk]
        all_top.append(cidx[order])
    return all_top

base_idx = cat_rerank_search(q_clip, g_clip, q_color, g_color, query_cats, gallery_cats)
r_base = evaluate(base_idx, q_pids, g_pids, '5.M.1 Phase 3 champion (CLIP+cat+color)')

# Text-only sanity check
text_idx = [np.argsort(-(g_text @ q_text[i]))[:K_TOP] for i in range(len(q_text))]
r_text = evaluate(text_idx, q_pids, g_pids, '5.M.2 Text-only (CLIP B/32 t2t)')

## Key finding: Text-only R@1=0.824, R@5=1.000 — perfect at K=5!
CLIP B/32 text embeddings (same model space as visual) are FAR more discriminative than cross-model L/14 text Anthony used.

## 5.M.3 Two-Stage: Visual top-K → Text Rerank

In [ ]:
def two_stage_search(q_vis, g_vis, q_txt, g_txt, q_col, g_col,
                     qcats, gcats, alpha_vis=0.40, k_retrieve=10):
    all_top = []
    for i in range(len(q_vis)):
        cat = qcats[i]
        cidx = np.where(gcats == cat)[0]
        clip_s  = g_vis[cidx] @ q_vis[i]
        color_s = g_col[cidx] @ q_col[i]
        fused1  = (1 - alpha_vis) * clip_s + alpha_vis * color_s
        top_k_local = np.argsort(-fused1)[:k_retrieve]
        top_k_cidx  = cidx[top_k_local]
        text_s  = g_txt[top_k_cidx] @ q_txt[i]
        order2  = np.argsort(-text_s)
        all_top.append(top_k_cidx[order2])
    return all_top

print('Two-stage visual->text (varying K):')
results_k = {}
for k in [5, 10, 20]:
    idx = two_stage_search(q_clip, g_clip, q_text, g_text, q_color, g_color,
                            query_cats, gallery_cats, alpha_vis=GLOBAL_ALPHA, k_retrieve=k)
    r = evaluate(idx, q_pids, g_pids, f'  Two-stage K={k:2d}')
    results_k[k] = r

## Observation: K=20 → R@1=0.9036 — breakthrough above 0.90!
Phase 4 showed R@20=0.9435 from visual alone, meaning text reranker has 90.4% hit rate on those candidates.

## 5.M.5 ABLATION: Which component matters most?

In [ ]:
# Load phase 5 results from disk (run by scripts/run_phase5_mark.py)
with open(RES / 'phase5_mark_results.json') as f:
    p5 = json.load(f)['phase5_mark']

ablation = p5['experiments']['5M5_ablation']
print('ABLATION RESULTS:')
print(f"{'System':<40} {'R@1':>6}  {'Delta':>8}")
print('-'*58)
systems = [
    ('Full (cat+CLIP+color+text)', 'full'),
    ('Remove text rerank', 'no_text'),
    ('Remove color', 'no_color'),
    ('Remove category filter', 'no_cat'),
    ('Remove CLIP visual', 'no_clip'),
]
for name, key in systems:
    r1 = ablation[key]['R@1']
    delta = ablation[key].get('delta_vs_full', 0)
    print(f'{name:<40} {r1:>6.4f}  {delta:>+8.4f}')

print()
print('COUNTERINTUITIVE: Removing CLIP visual IMPROVES R@1 (+0.0135)!')
print('When text descriptions are rich, visual embeddings = redundant noise.')

In [ ]:
# Visualization: All phases comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plot 1: R@1 progression across all phases
models_all = [
    ('ResNet50 (A P1)', 0.307, 'P1'),
    ('EffNet-B0 (M P1)', 0.369, 'P1'),
    ('ResNet+color (M P1)', 0.405, 'P1'),
    ('CLIP B/32 bare (M P2)', 0.480, 'P2'),
    ('CLIP B/32+color (M P2)', 0.576, 'P2'),
    ('CLIP L/14 (A P3)', 0.553, 'P3'),
    ('Text-only L/14 (A P3)', 0.602, 'P3'),
    ('CLIP L/14+color (A P3)', 0.642, 'P3'),
    ('CLIP B/32+cat+color (M P3)', 0.683, 'P3'),
    ('Per-cat oracle (M P4)', 0.695, 'P4'),
    ('Text-only B/32 (M P5)', 0.824, 'P5'),
    ('Two-stage K=10 (M P5)', 0.861, 'P5'),
    ('Three-stage best (M P5)', 0.9065, 'P5'),
    ('Cat+color+text (ablation)', 0.9200, 'P5'),
]
colors_map = {'P1': '#95a5a6', 'P2': '#3498db', 'P3': '#e67e22', 'P4': '#9b59b6', 'P5': '#27ae60'}
ax = axes[0]
ax.barh([m[0] for m in models_all], [m[1] for m in models_all],
         color=[colors_map[m[2]] for m in models_all], alpha=0.85)
ax.axvline(0.9065, color='red', linestyle='--', linewidth=1.5)
ax.set_xlabel('R@1'); ax.set_title('All Experiments R@1', fontsize=11, fontweight='bold')
ax.set_yticklabels([m[0] for m in models_all], fontsize=8)

# Plot 2: Ablation
abl_names = ['Full system', '-text rerank', '-color', '-category', '-CLIP visual']
abl_r1    = [0.9065, 0.6699, 0.8491, 0.8345, 0.9200]
abl_cols  = ['#27ae60', '#e74c3c', '#e74c3c', '#e74c3c', '#2ecc71']
axes[1].bar(abl_names, abl_r1, color=abl_cols, alpha=0.85)
axes[1].set_ylabel('R@1'); axes[1].set_title('Ablation Study', fontsize=11, fontweight='bold')
axes[1].set_xticklabels(abl_names, rotation=30, ha='right')
for i, v in enumerate(abl_r1): axes[1].text(i, v+0.01, f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')

# Plot 3: Two-stage K sweep
ks = [5, 10, 20]
r1_ks = [results_k[k]['R@1'] for k in ks]
axes[2].plot(ks, r1_ks, 'o-', color='#27ae60', linewidth=2.5, markersize=10)
axes[2].axhline(r_base['R@1'], color='gray', linestyle=':', label=f'Baseline R@1={r_base["R@1"]:.4f}')
axes[2].set_xlabel('K (visual candidates)'); axes[2].set_ylabel('R@1')
axes[2].set_title('Two-Stage K Sweep', fontsize=11, fontweight='bold')
axes[2].legend(); axes[2].grid(alpha=0.3)
for k, v in zip(ks, r1_ks): axes[2].annotate(f'{v:.4f}', (k, v), xytext=(0, 10), textcoords='offset points', ha='center')

plt.suptitle('Phase 5: Advanced Techniques — R@1=0.9065 (+23.7pp)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RES / 'phase5_mark_notebook_plots.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nSaved: results/phase5_mark_notebook_plots.png')

## Summary of Key Findings

| Finding | Value | Impact |
|---------|-------|--------|
| Text reranking gain | +23.7pp R@1 | Biggest single gain in the project |
| CLIP visual ablation | +1.35pp when removed | COUNTERINTUITIVE: text > visual |
| Category filter | -7.2pp when removed | Most critical structural component |
| Color histogram | -5.7pp when removed | Important visual complement to text |
| Best system R@1 | 0.9200 (cat+color+text) | 92% top-1 accuracy |

## LinkedIn Post Angle
"I accidentally found the best fashion visual search system by REMOVING my most expensive component.

After 5 phases building a visual product search engine:
- Phase 1-4: CLIP visual embeddings were the core signal
- Phase 5: Added text descriptions as a reranker
- Ablation: Removed CLIP visual

R@1 went UP from 0.906 to 0.920.

When product descriptions are paragraph-length and precise, text embeddings capture color, style, and cut better than visual embeddings do. The visual model sees light variation, pose changes, and background noise. The text says 'Black skinny jeans, zip fly, 79% cotton'.

The lesson: don't add components. Test if removing them helps."